# Data Source 06 · 경영·마케팅

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| Wikimedia Pageviews REST API | 무키 (User-Agent 필요) | A |
| UCI Online Retail II (id 502, CC BY 4.0) | 수동 | A |
| UCI Bike Sharing (id 275, CC BY 4.0) | 수동 | A |
| Google Trends (`pytrends`) | 비공식 | C (권장 안 함) |

기본 경로는 Wikimedia 일별 조회수(관심도 대리변수)이다. 소매 거래·대여량은 UCI에서 내려받아 별도 셀로 읽는다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_wikimedia_pageviews(obj):
    items = obj["items"]
    idx = pd.to_datetime([it["timestamp"][:8] for it in items], format="%Y%m%d")
    return pd.Series([it["views"] for it in items], index=idx, name="views")

In [ ]:
ARTICLE = "Time_series"
try:
    u = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/"
         f"{ARTICLE}/daily/20230101/20231231")
    y = parse_wikimedia_pageviews(json.loads(fetch_bytes(u)))
    DATA_KIND = "REAL: Wikimedia pageviews " + ARTICLE
except Exception as e:
    print("수집 실패 -> 합성 대체:", type(e).__name__, e)
    y = synthetic_series(365, "D", 2026061, trend=0.02, season=7, noise=2.0, start="2023-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND:", DATA_KIND); quality_report(y); y.plot(title=DATA_KIND); plt.show()

## 점검 체크리스트
- 조회수는 봇/자동 접근 필터(`user` 에이전트) 정의에 따라 달라진다
- 개입(캠페인·이벤트) 날짜가 외생적인지 사전에 정의했는가 (Ch.7)
- Online Retail II: 취소/반품 행(송장번호 `C` 접두 등)과 결측 고객ID 처리는 데이터 설명 페이지로 확인
- 개인정보·라이선스 조건 확인. 원자료는 재배포하지 않는다

## 추가 D. 국제선 항공 승객 (Box–Jenkins, 1949–1960 월별 144개)
- 학생 예제 07번이 쓰는 고전 교육용 시계열. **네트워크 없이** 저장소에 내장한 값을 쓰므로 항상 재현된다(원 예제의 내장 CSV와 동일 값, 첫 값 112, 마지막 값 432).
- 승법적 계절성(변동이 수준과 함께 커짐)이 있어 **로그 변환** 후 SARIMA/ETS를 쓰는 것이 표준이다. 표본이 작으므로(144) 외표본 결론은 신중히 해석한다.

In [ ]:
AIRLINE_CSV = """Month,Passengers
1949-01,112
1949-02,118
1949-03,132
1949-04,129
1949-05,121
1949-06,135
1949-07,148
1949-08,148
1949-09,136
1949-10,119
1949-11,104
1949-12,118
1950-01,115
1950-02,126
1950-03,141
1950-04,135
1950-05,125
1950-06,149
1950-07,170
1950-08,170
1950-09,158
1950-10,133
1950-11,114
1950-12,140
1951-01,145
1951-02,150
1951-03,178
1951-04,163
1951-05,172
1951-06,178
1951-07,199
1951-08,199
1951-09,184
1951-10,162
1951-11,146
1951-12,166
1952-01,171
1952-02,180
1952-03,193
1952-04,181
1952-05,183
1952-06,218
1952-07,230
1952-08,242
1952-09,209
1952-10,191
1952-11,172
1952-12,194
1953-01,196
1953-02,196
1953-03,236
1953-04,235
1953-05,229
1953-06,243
1953-07,264
1953-08,272
1953-09,237
1953-10,211
1953-11,180
1953-12,201
1954-01,204
1954-02,188
1954-03,235
1954-04,227
1954-05,234
1954-06,264
1954-07,302
1954-08,293
1954-09,259
1954-10,229
1954-11,203
1954-12,229
1955-01,242
1955-02,233
1955-03,267
1955-04,269
1955-05,270
1955-06,315
1955-07,364
1955-08,347
1955-09,312
1955-10,274
1955-11,237
1955-12,278
1956-01,284
1956-02,277
1956-03,317
1956-04,313
1956-05,318
1956-06,374
1956-07,413
1956-08,405
1956-09,355
1956-10,306
1956-11,271
1956-12,306
1957-01,315
1957-02,301
1957-03,356
1957-04,348
1957-05,355
1957-06,422
1957-07,465
1957-08,467
1957-09,404
1957-10,347
1957-11,305
1957-12,336
1958-01,340
1958-02,318
1958-03,362
1958-04,348
1958-05,363
1958-06,435
1958-07,491
1958-08,505
1958-09,404
1958-10,359
1958-11,310
1958-12,337
1959-01,360
1959-02,342
1959-03,406
1959-04,396
1959-05,420
1959-06,472
1959-07,548
1959-08,559
1959-09,463
1959-10,407
1959-11,362
1959-12,405
1960-01,417
1960-02,391
1960-03,419
1960-04,461
1960-05,472
1960-06,535
1960-07,622
1960-08,606
1960-09,508
1960-10,461
1960-11,390
1960-12,432"""
def load_airline():
    df = pd.read_csv(io.StringIO(AIRLINE_CSV))
    df.columns = ["month", "passengers"]
    s = pd.Series(df["passengers"].astype(float).values, index=pd.to_datetime(df["month"]), name="passengers").asfreq("MS")
    assert len(s) == 144 and s.isna().sum() == 0 and s.iloc[0] == 112 and s.iloc[-1] == 432
    return s
air = load_airline(); KIND_AIR = "REAL (embedded): Box-Jenkins airline passengers 1949-1960"
print("KIND_AIR:", KIND_AIR); quality_report(air)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5)); air.plot(ax=ax[0], title="수준 (변동이 커짐)"); np.log(air).plot(ax=ax[1], title="로그"); plt.tight_layout(); plt.show()